# 06 — Knowledge Graph Construction: Keyword Extraction

**Phase 6** (plan §6 Stage 6). Runs after `05_chunking_embeddings_bakeoff.ipynb`.

## What this notebook does

1. **Pre-flight check** — verifies Neo4j connectivity, CHUNK count, and schema indexes (including `keyword_name_unique` constraint and `keyword_embedding_index`).
2. **LLM smoke probe** — extracts keywords from 3 sample chunks and inspects JSON output (validates the prompt/parse pipeline before bulk run).
3. **Keyword extraction smoke run** — processes 50 chunks by default (`RUN_FULL=True` for all 34 K+), wires `(:KEYWORD)` nodes and `(:CHUNK)-[:MENTION]->(:KEYWORD)` relationships.
4. **KEYWORD statistics** — top-20 keywords by frequency, type distribution, average mentions per chunk.
5. **Neighborhood sample** — shows the keyword subgraph for a single chunk (chunk → keywords + sibling chunks sharing those keywords).
6. **Artefact write** — saves `notebooks/_artifacts/06_kg_construction/kg_construction.json`.
7. **Production runner note** — points to `scripts/run_keyword_extraction.py` for the full corpus overnight run.

## Keyword types

| Type | Description |
|---|---|
| `PERSON` | Historical figures (帝王、宰相、著者…) |
| `PLACE` | Geographic names (州、縣、域外地名…) |
| `DYNASTY` | Dynastic names (唐、隋、漢…) |
| `OFFICIAL_TITLE` | Official posts (刺史、御史、宰相…) |
| `EVENT` | Historical events (安史之亂、玄武門之變…) |
| `TEXT_TITLE` | Primary and secondary source titles (《唐律疏議》…) |
| `CONCEPT` | Administrative/philosophical concepts (均田制、科舉…) |
| `LEGAL_TERM` | Legal and regulatory terms (律、令、格、式…) |
| `ERA_NAME` | Reign era names (貞觀、開元…) |
| `OTHER` | Everything else |

## Graph spine addition

```
(:CHUNK)-[:MENTION {weight, mentionedAt}]->(:KEYWORD {name, type, frequency})
```

**Next**: Phase 7 — `07_search.ipynb` (vector similarity search + keyword-weighted re-ranking).


In [ ]:
import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

# --- repo root on sys.path
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "06_kg_construction"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# --- smoke / full config
RUN_FULL  = os.environ.get("RUN_FULL", "").lower() in ("1", "true", "yes")
MAX_CHUNKS = None if RUN_FULL else int(os.environ.get("MAX_CHUNKS", "50"))
RECOMPUTE  = os.environ.get("RECOMPUTE", "").lower() in ("1", "true", "yes")

print(f"mode      : {'FULL CORPUS' if RUN_FULL else f'SMOKE (max_chunks={MAX_CHUNKS})'}")
print(f"recompute : {RECOMPUTE}")
print(f"artifact  : {ARTIFACT_DIR}")


---
## 1 — Pre-flight: connectivity + schema readiness


In [ ]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.graph.schema import init_schema

driver = get_driver()

with driver.session() as s:
    r = s.run("RETURN 1 AS ok").single()
    assert r["ok"] == 1, "Neo4j connection failed"
print("Neo4j: OK")

# Ensure Phase-6 indexes exist (keyword_name_unique, keyword_type_index, etc.)
schema_result = init_schema(driver)
print(f"Schema init: {schema_result.get('constraints_created', 0)} constraints, "
      f"{schema_result.get('lookup_indexes_created', 0)} lookup indexes, "
      f"{schema_result.get('vector_indexes_created', 0)} vector indexes")

# Pipeline readiness
with driver.session() as s:
    stats = s.run("""
        MATCH (c:CHUNK)
        RETURN
          count(c)                                                         AS chunks_total,
          count(CASE WHEN c.charCount >= 50 THEN 1 END)                   AS chunks_eligible,
          count(CASE WHEN c.embeddingStatus = 'ok' THEN 1 END)            AS chunks_embedded,
          count(CASE WHEN c.mentionStatus = 'ok' THEN 1 END)              AS chunks_with_keywords
    """).single()
    kw_count = s.run("MATCH (k:KEYWORD) RETURN count(k) AS n").single()
    mention_count = s.run("MATCH ()-[m:MENTION]->() RETURN count(m) AS n").single()

print()
print(f"CHUNK nodes total       : {stats['chunks_total']:,}")
print(f"CHUNK eligible (≥50 ch) : {stats['chunks_eligible']:,}")
print(f"CHUNK embedded          : {stats['chunks_embedded']:,}")
print(f"CHUNK with keywords     : {stats['chunks_with_keywords']:,}")
print(f"KEYWORD nodes           : {kw_count['n']:,}")
print(f"MENTION relationships   : {mention_count['n']:,}")

# Silra API probe
from apps.backend.llm.silra import ping
health = ping()
print()
print(f"Silra API ok   : {health['ok']}")
print(f"Chat model     : {health['chat_model']}")
if health.get('errors'):
    print(f"Silra errors   : {health['errors']}")


---
## 2 — LLM smoke probe: keyword extraction on 3 sample chunks


In [ ]:
from apps.backend.pipeline.keywords import extract_keywords_for_chunk
from apps.backend.llm.silra import get_silra_client

client = get_silra_client()

# Pick 3 diverse chunks: zh-classical, zh-modern, kanbun
with driver.session() as s:
    sample_chunks = s.run("""
        MATCH (p:PAGE)-[:HAS]->(c:CHUNK)
        WHERE p.language IN ['zh-classical', 'zh-modern', 'kanbun']
          AND c.charCount >= 100
          AND c.text IS NOT NULL
        WITH p.language AS lang, c
        ORDER BY lang, rand()
        WITH lang, collect(c)[0] AS sample
        RETURN sample.id AS chunk_id, sample.text AS text, lang
    """).data()

print(f"Probing {len(sample_chunks)} sample chunks...\n")
probe_results = []
for row in sample_chunks:
    chunk_id = row['chunk_id']
    lang = row['lang']
    text_preview = (row['text'] or '')[:100].replace('\n', ' ')
    print(f"--- [{lang}] {chunk_id}")
    print(f"    text: {text_preview}…")
    try:
        kws = extract_keywords_for_chunk(row['text'], client=client)
        for kw in kws:
            print(f"    {kw['type']:18s} {kw['name']}  (w={kw['weight']:.2f})")
        probe_results.append({'chunk_id': chunk_id, 'lang': lang, 'keywords': kws, 'status': 'ok'})
    except Exception as exc:
        print(f"    ERROR: {exc}")
        probe_results.append({'chunk_id': chunk_id, 'lang': lang, 'keywords': [], 'status': 'failed', 'error': str(exc)})
    print()

ok_count = sum(1 for r in probe_results if r['status'] == 'ok')
print(f"Probe result: {ok_count}/{len(probe_results)} OK")
assert ok_count > 0, "LLM probe failed — check Silra API key and model config before running full extraction"


---
## 3 — Keyword extraction run

Default: 50 chunks (smoke). Set `RUN_FULL=True` (cell 1) or `RUN_FULL=1` env var for full corpus.

**Estimated runtime**:
- Smoke (50 chunks): ~1–2 min (≈1.5 s/chunk round-trip to Silra)
- Full corpus (34 K chunks): ~14–16 hours — use `scripts/run_keyword_extraction.py` instead.


In [ ]:
import time
from apps.backend.pipeline.keywords import run_keyword_extraction

print(f"Starting keyword extraction: max_chunks={MAX_CHUNKS}, recompute={RECOMPUTE}")
print("(each chunk = 1 LLM call; progress logged below)")
print()

t0 = time.time()
report = run_keyword_extraction(
    driver,
    max_chunks=MAX_CHUNKS,
    recompute=RECOMPUTE,
)
elapsed = time.time() - t0

print()
print("=" * 50)
print(f"Extraction complete in {elapsed:.1f}s")
print(f"  Chunks processed : {report.chunks_total:,}")
print(f"  Chunks ok        : {report.chunks_ok:,}")
print(f"  Chunks failed    : {report.chunks_failed:,}")
print(f"  Chunks skipped   : {report.chunks_skipped:,}")
print(f"  Keywords extracted: {report.keywords_extracted:,}")
print(f"  KEYWORD nodes (total): {report.keywords_unique:,}")
if report.errors:
    print(f"  First 5 errors:")
    for e in report.errors[:5]:
        print(f"    {e}")


---
## 4 — KEYWORD statistics


In [ ]:
from apps.backend.pipeline.keywords import keyword_summary

summary = keyword_summary(driver)

print("=== Corpus-wide keyword extraction state ===")
print(f"  CHUNK mentionStatus=ok      : {summary['chunks_ok']:,}")
print(f"  CHUNK mentionStatus=failed  : {summary['chunks_failed']:,}")
print(f"  CHUNK mentionStatus=skipped : {summary['chunks_skipped']:,}")
print(f"  KEYWORD nodes total         : {summary['keywords_total']:,}")
print()

print("=== Type distribution ===")
for row in summary['type_distribution']:
    bar = '█' * min(40, int(row['n'] / max(summary['keywords_total'], 1) * 400))
    print(f"  {(row['type'] or 'None'):18s} {row['n']:6,}  {bar}")
print()

print("=== Top 20 keywords by frequency ===")
for i, row in enumerate(summary['top_keywords'], 1):
    print(f"  {i:3d}. [{row['type']:18s}] {row['name']:20s}  freq={row['freq']}")


---
## 5 — MENTION relationship stats


In [ ]:
with driver.session() as s:
    mention_stats = s.run("""
        MATCH (c:CHUNK)-[m:MENTION]->(k:KEYWORD)
        RETURN
          count(m)            AS total_mentions,
          count(DISTINCT c)   AS chunks_with_mentions,
          count(DISTINCT k)   AS keywords_mentioned,
          avg(m.weight)       AS avg_weight,
          avg(c.mentionKeywordCount) AS avg_kw_per_chunk
    """).single()

    # Weight distribution
    weight_dist = s.run("""
        MATCH ()-[m:MENTION]->()
        WITH round(m.weight * 10) / 10 AS bucket, count(*) AS n
        RETURN bucket, n ORDER BY bucket
    """).data()

    # Most connected keywords (highest in-degree)
    hub_kws = s.run("""
        MATCH (c:CHUNK)-[:MENTION]->(k:KEYWORD)
        WITH k, count(DISTINCT c) AS chunk_count
        ORDER BY chunk_count DESC
        LIMIT 15
        RETURN k.name AS name, k.type AS type, chunk_count
    """).data()

print("=== MENTION relationship overview ===")
if mention_stats['total_mentions']:
    print(f"  Total MENTION rels        : {mention_stats['total_mentions']:,}")
    print(f"  Chunks with mentions      : {mention_stats['chunks_with_mentions']:,}")
    print(f"  Unique KEYWORD nodes used : {mention_stats['keywords_mentioned']:,}")
    print(f"  Avg weight per mention    : {mention_stats['avg_weight']:.3f}")
    print(f"  Avg keywords per chunk    : {mention_stats['avg_kw_per_chunk']:.1f}")
    print()

    print("=== MENTION weight distribution ===")
    for row in weight_dist:
        bar = '█' * min(40, int(row['n'] / max(mention_stats['total_mentions'], 1) * 400))
        print(f"  w={row['bucket']:.1f}  {row['n']:6,}  {bar}")
    print()

    print("=== Top 15 hub keywords (most chunks mentioning) ===")
    for i, row in enumerate(hub_kws, 1):
        print(f"  {i:3d}. [{row['type']:18s}] {row['name']:25s}  in {row['chunk_count']} chunks")
else:
    print("  No MENTION relationships yet — run the extraction first (cell 3).")


---
## 6 — Neighborhood sample: a chunk and its keyword cluster


In [ ]:
# Pick the chunk with the most keywords as the center of a mini subgraph.
with driver.session() as s:
    center = s.run("""
        MATCH (c:CHUNK)-[:MENTION]->(k:KEYWORD)
        WITH c, count(k) AS kcount
        ORDER BY kcount DESC
        LIMIT 1
        RETURN c.id AS chunk_id, c.text AS text, kcount
    """).single()

if not center:
    print("No MENTION relationships yet — run cell 3 first.")
else:
    chunk_id = center['chunk_id']
    print(f"Center chunk: {chunk_id}")
    print(f"Text preview: {(center['text'] or '')[:200].replace(chr(10), ' ')}")
    print(f"Keyword count: {center['kcount']}")
    print()

    with driver.session() as s:
        # Chunk's keywords
        kws = s.run("""
            MATCH (c:CHUNK {id: $cid})-[m:MENTION]->(k:KEYWORD)
            RETURN k.name AS name, k.type AS type, m.weight AS weight
            ORDER BY m.weight DESC
        """, cid=chunk_id).data()

        # Sibling chunks that share at least one keyword with this chunk
        siblings = s.run("""
            MATCH (c:CHUNK {id: $cid})-[:MENTION]->(k:KEYWORD)<-[:MENTION]-(sib:CHUNK)
            WHERE sib.id <> $cid
            WITH sib, count(DISTINCT k) AS shared
            ORDER BY shared DESC
            LIMIT 5
            RETURN sib.id AS sibling_id, shared
        """, cid=chunk_id).data()

    print("Keywords of center chunk:")
    for kw in kws:
        print(f"  [{kw['type']:18s}] {kw['name']:25s}  w={kw['weight']:.2f}")
    print()

    if siblings:
        print("Sibling chunks sharing keywords (top 5):")
        for sib in siblings:
            print(f"  {sib['sibling_id']}  (shared={sib['shared']})")
    else:
        print("No sibling chunks yet (smoke run too small — run full corpus to see cross-chunk links).")


---
## 7 — Artefact write


In [ ]:
artifact = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_full": RUN_FULL,
    "max_chunks": MAX_CHUNKS,
    "recompute": RECOMPUTE,
    "extraction_report": report.to_dict(),
    "summary": {
        "chunks_ok": summary['chunks_ok'],
        "chunks_failed": summary['chunks_failed'],
        "chunks_skipped": summary['chunks_skipped'],
        "keywords_total": summary['keywords_total'],
        "type_distribution": summary['type_distribution'],
        "top_keywords": summary['top_keywords'],
    },
}

out_path = ARTIFACT_DIR / "kg_construction.json"
out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"Artefact written to: {out_path}")
print(f"File size: {out_path.stat().st_size:,} bytes")


---
## 8 — Summary & production runner

### What was built

```
(:CHUNK {id, text, …})
    -[:MENTION {weight, mentionedAt}]->
(:KEYWORD {name, type, frequency, keywordAt})
```

- `KEYWORD.name` is the unique key (Traditional Chinese canonical form).
- `KEYWORD.frequency` is incremented on every MERGE so the most-cited entities rise to the top without a full graph scan.
- `CHUNK.mentionStatus` gates the pipeline (`'ok' | 'failed' | 'skipped'`).

### Production background runner (full corpus)

```bash
caffeinate -dimsu uv run python scripts/run_keyword_extraction.py \
    --batch-size 200 \
    --log-file logs/run_keyword_extraction.log
```

Estimated runtime: ~34 K chunks × 1.5 s/call ≈ **14–16 hours** on a single thread.  
Monitor: `tail -f logs/run_keyword_extraction.log`  
Report saved to: `logs/keyword_extraction_report.json`

### Next phase

**Phase 7** — `07_search.ipynb`
- Vector similarity search on `CHUNK.embedding` (cosine, 1024-dim).
- Keyword-weighted re-ranking: boost hits where query keywords overlap MENTION graph.
- Full verifier pipeline: citation retrieval → evidence scoring → answer synthesis.
